In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives']


In [3]:
from lib import allmusic
mio   = allmusic.MusicDBIO(verbose=False)
webio = allmusic.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant AllMusic DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AllMusic


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
searchArtists      = mio.data.getSearchArtistData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artists:             {0}".format(len(localArtists.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

AllMusic Search Results
   Global Master Search:      761013
   Global Master Search Data: 0
   Local Artists:             477570
   Errors:                    521
   Search Artists:            1803604
   Known Summary IDs:         718650


# Search For New Artists

In [ ]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [ ]:
useKnownData  = False
useMasterData = True

if useKnownData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].notna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  50696
#   Artist Names To Get:  42803
#   Artist Names To Get:  33587
#   Artist Names To Get:  21888
#   Artist Names To Get:  10630

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    if searchedForErrors.get(artistName) is not None:
        continue

    try:
        response = webio.getArtistSearchData(artistName=artistName)
    except:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(60)
        continue
        
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,list):
                for item in searchTermData:
                    if isinstance(item,dict):
                        artistID = item['id'][2:] if isinstance(item.get('id'),str) else None
                        if artistID is not None:
                            searchData[artistID] = {k: v for k,v in item.items() if k in ['name','ref']}
        df = DataFrame(searchData).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = allmusic.MusicDBIO(local=False).data.getSearchArtistData()
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(searchDF[searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} New Artists".format(searchDF[~searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    allmusic.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)
    
#Found 1628828 Unique Results
#Found 1639245 Unique Results
#Found 1664572 Unique Results
#Found 1674908 Unique Results

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [6]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [59]:
useKnown = False

artistNames       = searchArtists
localArtistsDict  = localArtists.get()
artistNamesToGet  = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useKnown is True:
    pdbio = PanDBIO()
    pdbio.setData()
    matchedIDs = pdbio.getNotNaDBIDs(db)[db]
    artistNamesToGet = artistNamesToGet[artistNamesToGet.index.isin(matchedIDs)]
    del pdbio

print("# {0} Search Results".format(db))
print("#   Available Names:      {0}".format(len(artistNames)))
print("#   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))

#   Artist Names To Get:  1109384
#   Artist Names To Get:  1093759
#   Artist Names To Get:  1085234
#   Artist Names To Get:  1070309
#   Artist Names To Get:  1047009

# AllMusic Search Results
#   Available Names:      1803604
#   Known Artist Names:   776595
#   Artist Names To Get:  1027009


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
tt = TermTime("today", "8:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["name"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting AllMusic ArtistIDs] Start    ==> Time Is 2022-05-23 07:07:20
========================= termTime(day=today,time=8:00pm) =========================
   ====> Terminate Time Set To 2022-05-23 20:00:00 <====
   ====> Will Terminate Process 12 Hours and 52 Minutes From Now
Getting Garreth Broesche (0002318150)                     True
Getting Garreth Hughes (0002438146)                       True
Getting Garreth Neilly (0002936072)                       True
Getting Garreth Asher (0003411790)                        True
Getting Garreth Gibson (0004140576)                       True
Getting Lyrical Assassins (0000171618)                    True
Getting Lyrical Desporados (0000208963)                   True
Getting Lyrical Eaze (0000399748)                         True
Getting Lyrical Twister (0000449083)                      True
Getting Lyrical Lizard (0000474021)                       True
Getting Bad Reputation (0002006446)                       True
Getting Solid Reputatio

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Teresa Peruzzi (0003806422)                       True
Getting Peruzzi (0003845926)                              True
Getting Edio Marcos De Souza (0001550044)                 True
Getting Luca Peruzzi (0000287764)                         True
Getting G. Peruzzi (0001094631)                           True
Getting Simone Peruzzi (0002384906)                       True
Getting Aurelio Peruzzi (0002488942)                      True
Getting John Peruzzi (0002635595)                         True
Getting Gianluca Peruzzi (0002832305)                     True
Getting Marino Peruzzi (0003004173)                       True
Getting Daniele Peruzzi (0003411187)                      True
Getting Francesco Peruzzi (0003664470)                    True
Getting Edmundo Peruzzi (0003897767)                      True
Getting Mattia Peruzzi (0004017316)                       True
Getting Matt Peruzzi (0004041995)                         True
Getting Dario Mafia (0003090465)                       

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Cool Country Band (0001556093)                    True
Getting The Cool Yule Band (0001777590)                   True
Getting The Healing Waters Band (0000875037)              True
Getting Harry Waters Band (0001595524)                    True
Getting The Continental Five Richmonds Lil Waters Band (0004206659)True
Getting The Cool Band (0000269232)                        True
Getting H.H. Waters' Goodtime Country Band (0000517169)   True
Getting The Secrets We Keep (0003283526)                  True
Getting Keep Company (0002627107)                         True
Getting Nadell Evans (0001325190)                         True
Getting Jism (0001273362)                                 True
Getting Grupo Manada (0000540127)                         True
Getting Grupo Monte Azul (0000856591)                     True
Getting Grupo Monti Agarro (0002104816)                   True
Getting Groupement Culturel Renault (0003835313)          True
Getting Groupement Rituel D'Instruments a Vent

...

Getting Vicious (0000329396)                              True
Getting The Vicious (0001552974)                          True
Getting Vicious (0001920546)                              True
Getting Vicious (0002083677)                              True
Getting Vicious (0003485279)                              True
Getting Vicious (0004144130)                              True
Getting Vicious Storm (0000164535)                        True
Getting Vicious Vic (0000198106)                          True
Getting Raise the Kicks (0003571250)                      True
Getting Yvonne Doll & The Locals (0001337692)             True
Getting Claudia Phillips & the Kicks (0001689636)         True
Getting Nikki Doll & the Penny Drops (0003310041)         True
Getting Shea the Doll (0003072401)                        True
Getting Turn the Doll (0004032215)                        True
Getting Linda "The Cupid Doll" Brigette (0003131501)      True
Getting Maria Rose & the Swiss Kicks (0002790963)      

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Reese Karlan (0003185236)                         True
Getting Back Burner (0000073422)                          True
Getting Matthew Landreth (0002981674)                     True
Getting Gonna Fall Hard (0001988867)                      True
Getting Heaven Falls Hard (0002115604)                    True
Getting Full Moon Heart (0000186274)                      True
Getting Phil Hurtt (0000338022)                           True
Getting Heart Full of Dirt (0000703845)                   True
Getting Phil Hart (0001052401)                            True
Getting Phil Hardy (0001249578)                           True
Getting Phil Hurt (0001394017)                            True
Getting Hardy Veles (0002531430)                          True
Getting Val Heart (0002967776)                            True
Getting Phyllis Hardy (0003208338)                        True
Getting Great Heights Band (0003440884)                   True
Getting Such Great Heights (0003378102)                

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Al Hellbound (0002567797)                         True
Getting New Memphis Hepcats (0002097593)                  True
Getting Daisy Mae & Hepcats (0000563709)                  True
Getting J. Mikel & The Hepcats (0000106113)               True
Getting Daisy Mae & Her Hepcats (0000671071)              True
Getting Street Pharmacy/Magno (0002055618)                True
Getting Directions (0000098485)                           True
Getting The Directions (0000123986)                       True
Getting Directions (0000166587)                           True
Getting Directions (0002009897)                           True
Getting Directions in Groove (0000141054)                 True
Getting The Bad Directions (0002075023)                   True
Getting Loft 55 (0000120221)                              True
Getting Helen 55 (0000567106)                             True
Getting Fortification 55 (0000660238)                     True
Getting Fortification 55 (0001500524)                  

In [ ]:
from lib.allmusic import moveLocalFiles

In [ ]:
moveLocalFiles()